In [1]:
# IMPORTS
import sys
import os
import torch
import torch.nn as nn
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from datetime import datetime as dt

import pandas, anndata, torch
print(pandas.__version__, anndata.__version__, torch.__version__)

# IMPORT OWN FILES
sys.path.append(os.path.abspath(".."))
import data_handling as dh
import binn_training as bt
from binn import BINN
import fold_prep as fp

# Supress warnings
import warnings
warnings.filterwarnings("ignore", message="Series.__getitem__ treating keys as positions", category=FutureWarning)
warnings.filterwarnings("ignore", message="The class AnnLoader is deprecated", category=FutureWarning)

C:\Users\nordm\AppData\Local\Temp\ipykernel_10836\1345321289.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(pandas.__version__, anndata.__version__, torch.__version__)


2.3.3 0.13.4 2.14.1+cpu


In [2]:
import importlib.util
from pathlib import Path

# Determine parent folder
base_dir = Path.cwd().parent
path = base_dir / "pipeline_paths_mnt.py"

# Import module dynamically
spec = importlib.util.spec_from_file_location("ppaths", path)
ppaths = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ppaths)

# to keep consistent paths
run_vars = "mg_200_mc_200"
pp = ppaths.PipelinePaths("dev_local", pseudo_non_hvg_run_vars=run_vars)

base path found (D:\BBTX11-VT26-07\simulated): True
data path found (D:\BBTX11-VT26-07\simulated\data): True


In [6]:
# GLOBALS
TRAIN_SIZE = 0.8
BATCH_SIZE = 32
MAX_EPOCHS = 20
FOLDS_OUTER = 3
FOLDS_INNER = 3
N_TRIALS = 30
EVAL_METRIC = 'roc_auc'
N_HVGS = 1000
N_LAYERS = 4

# PARAMETERS
LR = 0.002204
WEIGHT_DECAY = 0.021469
DROPOUT = 0.176345
ACTIVATION_FN = nn.Tanh()

# PATHS
data_path = pp.compl_hvg_from_cv

# OTHER
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TUNE_HYPERPARAMETERS = True

now = dt.now().strftime("%y%m%d_%H%M")
if TUNE_HYPERPARAMETERS:
    now_hyper = now
else:
    # hyperparameters to read
    now_hyper = '261110_0958'

cell_types = ['test1']

all_results = {}
for cell_type in cell_types:
    print(f'RUNNING BINN PIPELINE FOR {cell_type}')
    data_info = f'{cell_type}_{run_vars}_{N_HVGS}HVGSfromCV'
    HPARAM_PATH = pp.binn_model_path / f"{now_hyper}_best_hyperparameters_{data_info}.json"

    print("Reading train and test data...")
    train_adata = ad.read_h5ad(data_path / f'{cell_type}_train_preprocessed_noHVG.h5ad')
    test_adata = ad.read_h5ad(data_path / f'{cell_type}_test_preprocessed_noHVG.h5ad')

    print(f'train_adata:\n{train_adata}')

    conns = fp.load_pathway_conns(pp.pathway_data_path / 'binn_connectivity.csv')

    data, masks, artifacts = fp.preprocess_split(
        train_adata,
        {'test': test_adata},
        N_HVGS,
        conns,
        N_LAYERS
    )

    train_loader, test_loader = dh.create_dataloaders(
        data['train'], data['test'], batch_size=BATCH_SIZE)

    dh.poison_scanner(train_loader, device)
    dh.poison_scanner(test_loader, device)
    
    # CROSS VALIDATION
    outer_scores, outer_params = fp.run_CV_nested_HVG_sel(
        train_adata,                           
        device,
        eval_metric=EVAL_METRIC,
        conns=conns,
        n_hvgs=N_HVGS,
        n_layers=N_LAYERS,
        k_outer=FOLDS_OUTER,
        k_inner=FOLDS_INNER,
        epochs=MAX_EPOCHS,
        n_trials=N_TRIALS,
        activation_fn=ACTIVATION_FN,
        save_path=pp.binn_model_path,
        data_info=data_info,
        time=now
        )
    
    # save masks
    mask_dir = pp.mask_hvg_from_cv / run_vars
    mask_dir.mkdir(parents=True, exist_ok=True)
    for i, (name, m) in enumerate(masks.items()):
        m.to_csv(mask_dir / f"{cell_type}_train_preprocessed_layer_{i}_mask.csv")

    # save processed datasets
    for split, a in data.items():
        a.write_h5ad(pp.compl_hvg_from_cv / f"{cell_type}_{split}_preprocessed.h5ad")

    # save HVGs also as plain text, readable without Python
    pd.Series(artifacts['hvgs'], name='gene').to_csv(
    pp.binn_model_path / f"{now}_hvgs_{data_info}.csv", index=False)

    _, in_features, layers_list, tensor_masks = dh.compute_features(masks, device)

    # Hyperparameter tuning
    print("Performing hyperparameter tuning...")
    final = dh.hyperparameter_tuning_optuna(
        data['train'], 
        in_features, 
        layers_list, 
        tensor_masks, 
        device, 
        batch_size=BATCH_SIZE, 
        activation_fn=ACTIVATION_FN,
        k=FOLDS_INNER, 
        epochs=MAX_EPOCHS,
        metric=EVAL_METRIC
        )
    print(f"Best parameters for BINN: {final}")
    dh.save_hyperparameters(final, HPARAM_PATH)

    artifact_path = pp.binn_model_path / f"{now}_artifacts_{data_info}.joblib"
        
    joblib.dump({
        **artifacts,
        'n_hvgs': N_HVGS,
        'n_layers': N_LAYERS,
        'eval_metric': EVAL_METRIC,
        'hparams': final,
        'run_vars': run_vars,
        'time': now,
        }, artifact_path)


    # model is MLP with same layers as BINN 
    binn, criterion, optimizer, scheduler = dh.create_model(
        in_features, 
        layers_list, 
        tensor_masks, 
        device, 
        lr=final["lr"], 
        weight_decay=final["weight_decay"], 
        dropout=final["dropout"], 
        epochs=final['tune_total_epochs'],
        activation_fn=ACTIVATION_FN
        )
    print(binn)

    print(f"criterion: {criterion},\n optimizer: {optimizer}")

    # Run it
    dead_nodes_dict = dh.find_dead_outputs(binn, tensor_masks)
    print(dead_nodes_dict)

    # Dead output
    dh.show_dead_nodes(dead_nodes_dict, masks)

    for batch in train_loader:
        print(batch.X)
        break

    history = dh.train_fixed_epochs(
            binn, 
            train_loader, 
            criterion=criterion, 
            optimizer=optimizer, 
            scheduler=scheduler, 
            device=device,
            n_epochs=final['epochs'],
            save_path=pp.binn_model_path,
            data_info=data_info,
            time=now,
            monitor_loader=test_loader
            )

    # Harvest the results
    probs, targets, test_auc = dh.evaluate_model_roc(binn, test_loader, device)
    print(f"Test ROC-AUC: {test_auc:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    # Loss plot
    axes[0].plot(history['train_loss'], label='Training Loss', color='royalblue', linewidth=2)
    axes[0].plot(history['test_loss'], label='Testing Loss', color='orange', linestyle='--', linewidth=2)
    axes[0].set_title('Loss', fontsize=14)
    axes[0].set_xlabel('Epochs', fontsize=12)
    axes[0].set_ylabel('Loss (BCE)', fontsize=12)
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Accuracy plot
    axes[1].plot(history['train_acc'], label='Training Accuracy', color='green', linewidth=2)
    axes[1].plot(history['test_acc'], label='Testing Accuracy', color='red', linestyle='--', linewidth=2)
    axes[1].set_title('Accuracy', fontsize=14)
    axes[1].set_xlabel('Epochs', fontsize=12)
    axes[1].set_ylabel('Accuracy', fontsize=12)
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle('BINN Training Diagnostics', fontsize=16)
    plt.tight_layout()
    fig.savefig(pp.binn_test_results_path / f'{now}_training_curves_{data_info}.png')
    plt.show()

    df_res = dh.save_test_results(
        binn, 
        test_loader, 
        device, 
        data_info, 
        pp.binn_test_results_path,
        now
        )
    
    dh.confusion_matrix_binn(df_res, pp.binn_test_results_path, data_info, now)

    report = dh.class_report(df_res)
    print(report)

    all_results[cell_type] = {
        'eval_metric': EVAL_METRIC,
        'nested_cv_mean': np.mean(outer_scores), 
        'nested_cv_std': np.std(outer_scores),
        'outer_epochs': [p['epochs'] for p in outer_params], 
        'final_epochs': final['epochs'],
    }
    
    pd.DataFrame(history).to_csv(
        pp.binn_model_path/ f'{now}_history_{data_info}.csv', 
        index_label='epoch')
    
    with open(pp.binn_test_results_path / f'{now}_class_report_{data_info}.txt', 'w') as f:
        f.write(report)

    print(f'COMPLETED BINN PIPELINE FOR {cell_type} \n\n\n')

summary = pd.DataFrame(all_results).T
summary.to_csv(pp.binn_test_results_path / f'{now}_summary_{run_vars}.csv')
summary



RUNNING BINN PIPELINE FOR test1
Reading train and test data...
train_adata:
AnnData object with n_obs × n_vars = 195 × 9293
    obs: 'subject', 'msex', 'age_death', 'pmi', 'race', 'Pathologic_diagnosis_of_AD', 'cogdx', 'braaksc', 'ceradsc', 'label', 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito'
    var: 'n_cells', 'mito', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    layers: None (.X)
Nr genes in dataset 9293 -> 1000
Aggregating single-cell data to pseudobulk by mean...


C:\Python314\Lib\functools.py:982: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
C:\Python314\Lib\functools.py:982: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Shape of pseudobulk AnnData: (6, 1000)

Nr genes in dataset 9293 -> 1000
Aggregating single-cell data to pseudobulk by mean...
Shape of pseudobulk AnnData: (2, 1000)

Standardizing data (zero mean, unit variance)...
Standardizing test data using training statistics...
Mapping 1000 genes to Reactome-paths...
Pathways pruned (unreachable or too deep): 749
Non-terminal pathways with gene conns: 1115
      ...calculating matrix for layer 0
      ...calculating matrix for layer 1
      ...calculating matrix for layer 2
      ...calculating matrix for layer 3
      ...calculating matrix for layer 4
Rolling up 'train'...
Rolling up 'test'...
Overlapping genes kept: 939 for train
Overlapping genes kept: 939 for test
No poison found!
No poison found!
Nr genes in dataset 9293 -> 1000
Aggregating single-cell data to pseudobulk by mean...
Shape of pseudobulk AnnData: (4, 1000)

Nr genes in dataset 9293 -> 1000
Aggregating single-cell data to pseudobulk by mean...


d:\BBTX11-VT26-07\venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
C:\Python314\Lib\functools.py:982: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
C:\Python314\Lib\functools.py:982: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Shape of pseudobulk AnnData: (2, 1000)

Standardizing data (zero mean, unit variance)...
Standardizing test data using training statistics...
Mapping 1000 genes to Reactome-paths...
Pathways pruned (unreachable or too deep): 738
Non-terminal pathways with gene conns: 1103
      ...calculating matrix for layer 0
      ...calculating matrix for layer 1
      ...calculating matrix for layer 2


[I 2026-11-17 21:16:50,202] A new study created in memory with name: no-name-47f2f89f-cc0a-45a2-8a18-a5588216ae66


      ...calculating matrix for layer 3
      ...calculating matrix for layer 4
Rolling up 'train'...
Rolling up 'val'...
Overlapping genes kept: 940 for train
Overlapping genes kept: 940 for val
input features: 940
layer list: [792, 451, 168, 29, 1]
Starting Optuna Hyperparameter Search...


d:\BBTX11-VT26-07\venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
d:\BBTX11-VT26-07\venv\Lib\site-packages\sklearn\metrics\_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
d:\BBTX11-VT26-07\venv\Lib\site-packages\sklearn\metrics\_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
d:\BBTX11-VT26-07\venv\Lib\site-packages\sklearn\metrics\_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
d:\BBTX11-VT26-07\venv\Lib\site-packages\sklearn\metrics\_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
d:\BBTX11-VT26-07\venv\Lib\s

KeyboardInterrupt: 